In [24]:
import pandas as pd
import numpy as np

import seaborn as sns
from matplotlib import pyplot as plt

from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer


from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import accuracy_score

In [25]:
df = pd.read_csv('zomato.csv')

In [26]:
df = df[df['location'] != 'Peenya'] 

### Drop less important columns

In [27]:
df.drop(columns=['url','address','name','votes','phone','dish_liked','cuisines','reviews_list','menu_item'],inplace=True)

### Data cleaning

In [28]:
df.isnull().sum()

online_order                      0
book_table                        0
rate                           7775
location                         21
rest_type                       227
approx_cost(for two people)     346
listed_in(type)                   0
listed_in(city)                   0
dtype: int64

In [29]:
df.duplicated().sum()
df.drop_duplicates(inplace=True)

In [30]:
df.dropna(inplace=True)

In [31]:
df= df.rename(columns={'approx_cost(for two people)':'cost','listed_in(type)':'type',
                                  'listed_in(city)':'city'})

In [32]:
olddf = df[(df['rate'] != "NEW") & (df['rate'] != "-")]

In [33]:
newdf = df[(df['rate'] == "NEW") | (df['rate'] == "-")]

### Rate column modifications

In [34]:
olddf['rate'] = olddf['rate'].str.replace('/5', '', regex=False).astype(float)

C:\Users\adeeb\AppData\Local\Temp\ipykernel_26644\1815034976.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  olddf['rate'] = olddf['rate'].str.replace('/5', '', regex=False).astype(float)


### Cost column modification

In [36]:
olddf['cost'] = olddf['cost'].str.replace(',', '').astype(int)

C:\Users\adeeb\AppData\Local\Temp\ipykernel_26644\800913028.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  olddf['cost'] = olddf['cost'].str.replace(',', '').astype(int)


### Column Transformer

In [37]:
preprocessor = ColumnTransformer(
    transformers=[
        ('ohe', OneHotEncoder(drop='first'), ['online_order', 'book_table', 'location','rest_type','type','city']),
    ],
    remainder='passthrough'
)

In [38]:
x = olddf.drop('rate' , axis =1)
y = olddf['rate']

### Splitting data into train and test

In [39]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=.15,random_state=353)

In [40]:
x_train.head()

,online_order,book_table,location,rest_type,cost,type,city
43767,No,Yes,Infantry Road,Casual Dining,1100,Dine-out,MG Road
1333,Yes,No,Bannerghatta Road,Food Court,250,Delivery,Bannerghatta Road
10220,No,No,BTM,Sweet Shop,300,Desserts,BTM
15472,Yes,No,Frazer Town,Quick Bites,350,Dine-out,Frazer Town
48020,Yes,No,Brigade Road,"Sweet Shop, Quick Bites",400,Desserts,Residency Road


In [41]:
x_test.head()

,online_order,book_table,location,rest_type,cost,type,city
46980,No,No,Malleshwaram,Casual Dining,1500,Dine-out,Rajajinagar
27523,Yes,Yes,Koramangala 5th Block,"Casual Dining, Bar",1500,Delivery,Koramangala 4th Block
43010,No,No,Brigade Road,Quick Bites,200,Delivery,MG Road
44508,Yes,No,New BEL Road,Casual Dining,400,Dine-out,New BEL Road
49469,No,No,Whitefield,Quick Bites,200,Delivery,Sarjapur Road


In [42]:
X_train_prepared = preprocessor.fit_transform(x_train)
X_test_prepared  = preprocessor.transform(x_test)

In [43]:
X_train_prepared = X_train_prepared.toarray()
X_test_prepared = X_test_prepared.toarray()

In [44]:
X_train_df = pd.DataFrame(X_train_prepared, columns=preprocessor.get_feature_names_out())
X_test_df = pd.DataFrame(X_test_prepared, columns=preprocessor.get_feature_names_out())

In [45]:
X_train_df

,ohe__online_order_Yes,ohe__book_table_Yes,ohe__location_Banashankari,ohe__location_Banaswadi,ohe__location_Bannerghatta Road,ohe__location_Basavanagudi,ohe__location_Basaveshwara Nagar,ohe__location_Bellandur,ohe__location_Bommanahalli,ohe__location_Brigade Road,...,ohe__city_MG Road,ohe__city_Malleshwaram,ohe__city_Marathahalli,ohe__city_New BEL Road,ohe__city_Old Airport Road,ohe__city_Rajajinagar,ohe__city_Residency Road,ohe__city_Sarjapur Road,ohe__city_Whitefield,remainder__cost
0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1100.0
1,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,250.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,300.0
3,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,350.0
4,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,400.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32066,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,600.0
32067,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1200.0
32068,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1100.0
32069,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,400.0


### Model training

In [47]:
DTree=DecisionTreeRegressor(min_samples_leaf=.0001)
DTree.fit(X_train_df,y_train)
y_predict=DTree.predict(X_test_df)


In [51]:
threshold = 4

# Create binary “success/failure” arrays (1 = success, 0 = failure)
y_testDtree = (y_test > threshold).astype(int)
y_predDtree = (y_predict > threshold).astype(int)


In [52]:
accuracy = accuracy_score(y_testDtree, y_predDtree)
print("Accuracy:", accuracy)

Accuracy: 0.8954063604240282


### Function for checking accuracy of newly opened restaurants

In [53]:
def Predict_success(online, bookings, location, rest_type, cost, type, city):
    
    test_input = pd.DataFrame([[online, bookings, location, rest_type, cost, type, city]],
                              columns=['online_order', 'book_table', 'location', 'rest_type', 'cost', 'type', 'city'])

    test_input_processed = preprocessor.transform(test_input)
    y_predict = DTree.predict(test_input_processed)

    return y_predict

In [54]:
ans = Predict_success('Yes', 'No', 'Koramangala 6th Block', 'Quick Bites', 3000, 'Desserts', 'Koramangala 5th Block')

C:\Users\adeeb\anaconda3\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but DecisionTreeRegressor was fitted with feature names
  warnings.warn(


In [56]:
print("Success" if ans > 4 else "Failure")
print(ans)

Success
[4.34285714]
